## Training

In [2]:
import ast
import inspect
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 256
OUTPUT_DIR = Path("./results/xlm_roberta_detection")
TRAIN_FILE = Path("PL_train_translated_lapa.csv")
TEST_FILE = Path("PL_test_translated_lapa.csv")


def label_from_techniques(value: object) -> int:
    """Convert a CSV techniques value into a binary persuasion label."""
    if isinstance(value, list):
        return int(bool(value))
    if pd.isna(value):
        return 0
    if not isinstance(value, str):
        raise TypeError(f"Expected techniques to be a list or string, got {type(value).__name__}")
    if len(value) == 2:
        return 0

    techniques = ast.literal_eval(value)
    if not isinstance(techniques, list):
        raise ValueError(f"Expected a list in the techniques column, got: {value!r}")
    return int(bool(techniques))


def load_data(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required_columns = {"content_uk", "techniques"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{path} is missing required columns: {sorted(missing_columns)}")

    frame["text"] = frame["content_uk"].fillna("").astype(str)
    frame["label"] = frame["techniques"].map(label_from_techniques).astype("int64")
    return frame[["text", "label"]]


def tokenize_dataset(dataset: Dataset, tokenizer: AutoTokenizer) -> Dataset:
    def tokenize_batch(batch: dict[str, list[str]]) -> dict[str, object]:
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    return dataset.map(tokenize_batch, batched=True, remove_columns=["text"])


def compute_metrics(prediction: object) -> dict[str, float]:
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted_labels = logits.argmax(axis=-1)
    true_labels = prediction.label_ids

    return {
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "precision": precision_score(true_labels, predicted_labels, zero_division=0),
        "recall": recall_score(true_labels, predicted_labels, zero_division=0),
        "f1": f1_score(true_labels, predicted_labels, zero_division=0),
        "macro_f1": f1_score(true_labels, predicted_labels, average="macro", zero_division=0),
    }


def main() -> None:
    use_cuda = torch.cuda.is_available()
    print(f"CUDA Available: {use_cuda}")
    if use_cuda:
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")

    train_frame = load_data(TRAIN_FILE)
    test_frame = load_data(TEST_FILE)

    train_part, validation_part = train_test_split(
        train_frame,
        test_size=0.1,
        random_state=42,
        stratify=train_frame["label"],
    )
    train_dataset = Dataset.from_pandas(train_part.reset_index(drop=True), preserve_index=False)
    validation_dataset = Dataset.from_pandas(
        validation_part.reset_index(drop=True), preserve_index=False
    )
    test_dataset = Dataset.from_pandas(test_frame, preserve_index=False)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    train_dataset = tokenize_dataset(train_dataset, tokenizer)
    validation_dataset = tokenize_dataset(validation_dataset, tokenizer)
    test_dataset = tokenize_dataset(test_dataset, tokenizer)

    training_arguments = {
        "output_dir": str(OUTPUT_DIR),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 32,
        "gradient_accumulation_steps": 1,
        "dataloader_pin_memory": True if use_cuda else False,
        "fp16": use_cuda,
        "num_train_epochs": 4,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "report_to": "none",
    }

    strategy_argument = (
        "eval_strategy"
        if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
        else "evaluation_strategy"
    )
    training_arguments[strategy_argument] = "epoch"

    trainer_arguments = {
        "model": model,
        "args": TrainingArguments(**training_arguments),
        "train_dataset": train_dataset,
        "eval_dataset": validation_dataset,
        "compute_metrics": compute_metrics,
    }
    processor_argument = (
        "processing_class"
        if "processing_class" in inspect.signature(Trainer.__init__).parameters
        else "tokenizer"
    )
    trainer_arguments[processor_argument] = tokenizer

    trainer = Trainer(**trainer_arguments)

    trainer.train()
    test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
    print("\nTest metrics:")
    for metric, value in test_metrics.items():
        print(f"{metric}: {value:.4f}" if isinstance(value, (int, float)) else f"{metric}: {value}")

    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))


if __name__ == "__main__":
    main()

CUDA Available: True
Using GPU: Tesla T4


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

... (10 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
0.536439,0.489936,4,0.780435,0.735409,0.851351,0.789144,0.780060



Test metrics:
test_loss: 0.4899
test_accuracy: 0.7804
test_precision: 0.7354
test_recall: 0.8514
test_f1: 0.7891
test_macro_f1: 0.7801


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## error analysis

In [3]:
from pathlib import Path
import ast
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

MODEL1_PATH = Path("./results/xlm_roberta_detection")
TEST_FILE = Path("PL_test_translated_lapa.csv")
MAX_LENGTH = 256

def label_from_techniques(value: object) -> int:
    if isinstance(value, list): return int(bool(value))
    if pd.isna(value): return 0
    if not isinstance(value, str): return 0
    if len(value) == 2: return 0
    techniques = ast.literal_eval(value)
    return int(bool(techniques))

# 1. Load test data
test_df = pd.read_csv(TEST_FILE)
test_df["text"] = test_df["content_uk"].fillna("").astype(str)
test_df["label"] = test_df["techniques"].map(label_from_techniques).astype("int64")

# 2. Tokenize
tokenizer1 = AutoTokenizer.from_pretrained(MODEL1_PATH)
model1 = AutoModelForSequenceClassification.from_pretrained(MODEL1_PATH)

test_dataset = Dataset.from_pandas(test_df[["text", "label"]])
test_dataset = test_dataset.map(
    lambda x: tokenizer1(x["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"]
)

# 3. Predict
trainer1 = Trainer(model=model1)
predictions = trainer1.predict(test_dataset)
logits = predictions.predictions[0] if isinstance(predictions.predictions, tuple) else predictions.predictions
preds1 = np.argmax(logits, axis=-1)
probs1 = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

# 4. Attach predictions to DataFrame
test_df["pred_model1"] = preds1
test_df["prob_model1"] = probs1

# 5. Print Classification Report & Confusion Matrix
print("=== MODEL 1 (XLM-RoBERTa) ERROR ANALYSIS ===")
print("\nClassification Report:")
print(classification_report(test_df["label"], preds1, target_names=["Neutral (0)", "Persuasive (1)"]))

cm = confusion_matrix(test_df["label"], preds1)
print(f"Confusion Matrix:\nTN: {cm[0,0]} | FP: {cm[0,1]}\nFN: {cm[1,0]} | TP: {cm[1,1]}\n")

# 6. Save False Positives and False Negatives for manual review
false_positives1 = test_df[(test_df["label"] == 0) & (test_df["pred_model1"] == 1)]
false_negatives1 = test_df[(test_df["label"] == 1) & (test_df["pred_model1"] == 0)]

false_positives1.to_csv("model1_false_positives.csv", index=False)
false_negatives1.to_csv("model1_false_negatives.csv", index=False)

print(f"Saved {len(false_positives1)} False Positives to 'model1_false_positives.csv'")
print(f"Saved {len(false_negatives1)} False Negatives to 'model1_false_negatives.csv'")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/460 [00:00<?, ? examples/s]

=== MODEL 1 (XLM-RoBERTa) ERROR ANALYSIS ===

Classification Report:
                precision    recall  f1-score   support

   Neutral (0)       0.84      0.71      0.77       238
Persuasive (1)       0.74      0.85      0.79       222

      accuracy                           0.78       460
     macro avg       0.79      0.78      0.78       460
  weighted avg       0.79      0.78      0.78       460

Confusion Matrix:
TN: 170 | FP: 68
FN: 33 | TP: 189

Saved 68 False Positives to 'model1_false_positives.csv'
Saved 33 False Negatives to 'model1_false_negatives.csv'


##better model

In [10]:
import ast
import inspect
from pathlib import Path
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "xlm-roberta-large"
MAX_LENGTH = 256
OUTPUT_DIR = Path("./results/xlm_roberta_large_detection")
TRAIN_FILE = Path("PL_train_translated_lapa.csv")
TEST_FILE = Path("PL_test_translated_lapa.csv")

def label_from_techniques(value: object) -> int:
    if isinstance(value, list): return int(bool(value))
    if pd.isna(value): return 0
    if not isinstance(value, str): return 0
    if len(value) == 2: return 0
    techniques = ast.literal_eval(value)
    return int(bool(techniques))

def load_data(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame["text"] = frame["content_uk"].fillna("").astype(str)
    frame["label"] = frame["techniques"].map(label_from_techniques).astype("int64")
    return frame[["text", "label"]]

def compute_metrics(prediction: object) -> dict[str, float]:
    logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
    preds = logits.argmax(axis=-1)
    labels = prediction.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

def main() -> None:
    use_cuda = torch.cuda.is_available()
    print(f"Training Model 2 ({MODEL_NAME}) | CUDA Available: {use_cuda}")

    train_frame = load_data(TRAIN_FILE)
    test_frame = load_data(TEST_FILE)

    train_part, val_part = train_test_split(
        train_frame, test_size=0.1, random_state=42, stratify=train_frame["label"]
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    def tokenize_fn(ds):
        return ds.map(
            lambda x: tokenizer(x["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH),
            batched=True,
            remove_columns=["text"],
        )

    train_dataset = tokenize_fn(Dataset.from_pandas(train_part.reset_index(drop=True)))
    val_dataset = tokenize_fn(Dataset.from_pandas(val_part.reset_index(drop=True)))
    test_dataset = tokenize_fn(Dataset.from_pandas(test_frame))

    training_arguments = {
        "output_dir": str(OUTPUT_DIR),
        "learning_rate": 1.5e-5,
        "per_device_train_batch_size": 8,       # Reduced batch size for large model memory safety
        "per_device_eval_batch_size": 16,
        "gradient_accumulation_steps": 2,       # Effective batch size = 16
        "fp16": use_cuda,                        # Native FP16 works smoothly with XLM-RoBERTa
        "num_train_epochs": 4,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "report_to": "none",
    }
    strategy_arg = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
    training_arguments[strategy_arg] = "epoch"

    trainer = Trainer(
        model=model,
        args=TrainingArguments(**training_arguments),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        processing_class=tokenizer,
    )

    trainer.train()
    test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")

    print("\nModel 2 (XLM-RoBERTa-Large) Test Metrics:")
    for metric, val in test_metrics.items():
        print(f"{metric}: {val:.4f}" if isinstance(val, (int, float)) else f"{metric}: {val}")

    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))

if __name__ == "__main__":
    main()

Training Model 2 (xlm-roberta-large) | CUDA Available: True


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

... (10 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
0.977781,0.518765,4,0.795652,0.748062,0.869369,0.804167,0.795265



Model 2 (XLM-RoBERTa-Large) Test Metrics:
test_loss: 0.5188
test_accuracy: 0.7957
test_precision: 0.7481
test_recall: 0.8694
test_f1: 0.8042
test_macro_f1: 0.7953


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

##error analysis

In [12]:
from pathlib import Path
import ast
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

MODEL1_PATH = Path("./results/xlm_roberta_detection")
MODEL2_PATH = Path("./results/xlm_roberta_large_detection")
TEST_FILE = Path("PL_test_translated_lapa.csv")
MAX_LENGTH = 256

def label_from_techniques(value: object) -> int:
    if isinstance(value, list): return int(bool(value))
    if pd.isna(value): return 0
    if not isinstance(value, str): return 0
    if len(value) == 2: return 0
    techniques = ast.literal_eval(value)
    return int(bool(techniques))

# 1. Load data
test_df = pd.read_csv(TEST_FILE)
test_df["text"] = test_df["content_uk"].fillna("").astype(str)
test_df["label"] = test_df["techniques"].map(label_from_techniques).astype("int64")

# 2. Get predictions from Model 1 (Base)
tok1 = AutoTokenizer.from_pretrained(MODEL1_PATH)
mod1 = AutoModelForSequenceClassification.from_pretrained(MODEL1_PATH)
ds1 = Dataset.from_pandas(test_df[["text"]]).map(lambda x: tok1(x["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH), batched=True, remove_columns=["text"])
logits1 = Trainer(model=mod1).predict(ds1).predictions
preds1 = np.argmax(logits1[0] if isinstance(logits1, tuple) else logits1, axis=-1)

# 3. Get predictions from Model 2 (Large)
tok2 = AutoTokenizer.from_pretrained(MODEL2_PATH)
mod2 = AutoModelForSequenceClassification.from_pretrained(MODEL2_PATH)
ds2 = Dataset.from_pandas(test_df[["text"]]).map(lambda x: tok2(x["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH), batched=True, remove_columns=["text"])
logits2 = Trainer(model=mod2).predict(ds2).predictions
preds2 = np.argmax(logits2[0] if isinstance(logits2, tuple) else logits2, axis=-1)

# 4. Comparative Metrics Summary
labels = test_df["label"].values
metrics_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "Macro F1"],
    "XLM-RoBERTa-base": [
        accuracy_score(labels, preds1),
        precision_score(labels, preds1, zero_division=0),
        recall_score(labels, preds1, zero_division=0),
        f1_score(labels, preds1, zero_division=0),
        f1_score(labels, preds1, average="macro", zero_division=0),
    ],
    "XLM-RoBERTa-large": [
        accuracy_score(labels, preds2),
        precision_score(labels, preds2, zero_division=0),
        recall_score(labels, preds2, zero_division=0),
        f1_score(labels, preds2, zero_division=0),
        f1_score(labels, preds2, average="macro", zero_division=0),
    ]
})

print("=== FINAL MODEL COMPARISON SUMMARY ===")
print(metrics_summary.to_string(index=False))

# 5. Extract instances fixed by XLM-RoBERTa-large
test_df["pred_base"] = preds1
test_df["pred_large"] = preds2

fixed_false_positives = test_df[(test_df["label"] == 0) & (test_df["pred_base"] == 1) & (test_df["pred_large"] == 0)]
fixed_false_positives.to_csv("fixed_false_positives_by_large.csv", index=False)

print(f"\nXLM-RoBERTa-large successfully corrected {len(fixed_false_positives)} False Positives made by XLM-RoBERTa-base.")
print("Saved corrected samples to 'fixed_false_positives_by_large.csv'")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/460 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Map:   0%|          | 0/460 [00:00<?, ? examples/s]

=== FINAL MODEL COMPARISON SUMMARY ===
   Metric  XLM-RoBERTa-base  XLM-RoBERTa-large
 Accuracy          0.780435           0.795652
Precision          0.735409           0.748062
   Recall          0.851351           0.869369
       F1          0.789144           0.804167
 Macro F1          0.780060           0.795265

XLM-RoBERTa-large successfully corrected 17 False Positives made by XLM-RoBERTa-base.
Saved corrected samples to 'fixed_false_positives_by_large.csv'
